# 농사로 실내정원용 식물 API: 실내식물 10종

**준비**: 왼쪽 열쇠 아이콘(Secrets)에 이름 `NONGSARO_API_KEY`로 농사로 API 키를 넣고 노트북 액세스를 켜요.

검색 결과에서 첫 번째 식물이 원하는 식물이 아니면(예: 골드크레스트 품종 이름이 다를 때) 알려 주세요.


In [ ]:
# 농사로 실내정원용 식물 API로 실내식물 10종 정보 받기
# 준비: 코랩 열쇠 아이콘(Secrets)에 NONGSARO_API_KEY 저장 (농사로 API 키: api.nongsaro.go.kr 에서 신청)
import json, urllib.parse, urllib.request
import xml.etree.ElementTree as ET
from google.colab import userdata

KEY = userdata.get("NONGSARO_API_KEY").strip()
BASE = "http://api.nongsaro.go.kr/service/garden"
PLANTS = {21:"금전수", 22:"산세베리아", 23:"스킨답서스", 24:"몬스테라", 25:"해피트리",
          26:"벵갈고무나무", 27:"아이비", 28:"홍콩야자", 29:"관음죽", 30:"골드크레스트"}

def get_items(url):
    xml = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")
    root = ET.fromstring(xml)
    items = []
    for it in root.iter("item"):
        items.append({c.tag: (c.text or "").strip() for c in it})
    return items, root

results = {}
for pid, name in PLANTS.items():
    q = urllib.parse.quote(name)
    lst, root = get_items(f"{BASE}/gardenList?apiKey={KEY}&sType=sCntntsSj&sText={q}&numOfRows=20")
    print(f"\n[{pid} {name}] 검색 결과 {len(lst)}개")
    if not lst:
        msg = root.findtext(".//resultMsg")
        print("  결과 없음", msg or "")
        results[name] = {"plant_id": pid, "search": []}
        continue
    for c in lst:
        print("  ", c.get("cntntsNo"), c.get("cntntsSj"))
    first = lst[0]
    dtl, _ = get_items(f"{BASE}/gardenDtl?apiKey={KEY}&cntntsNo={first.get('cntntsNo')}")
    results[name] = {"plant_id": pid, "search": lst, "detail": dtl[0] if dtl else {}}

text = json.dumps(results, ensure_ascii=False, indent=2).replace(KEY, "REDACTED")  # 키가 파일에 남지 않게
open("nongsaro_garden_10.json", "w", encoding="utf-8").write(text)
print("\n완료: nongsaro_garden_10.json")


In [ ]:
from google.colab import files
files.download('nongsaro_garden_10.json')
